# BÁO CÁO THỰC NGHIỆM LAB 02: N-GRAM LANGUAGE MODELS, SMOOTHING & PERPLEXITY

Môn học: Xử lý ngôn ngữ tự nhiên và ứng dụng  
Học kỳ: Học kỳ I - 2026  
Sinh viên: Nguyễn Hồng Quân - 23001921  
Giảng viên / TA: Phạm Ngọc Hải  
Chủ đề: N-gram Language Models, Smoothing and Perplexity

---

Báo cáo này thực hiện đầy đủ các yêu cầu thực nghiệm từ **Mục 13** đến **Mục 23** theo tài liệu `W2.pdf`:
1. **Experiment 1 (Mục 13):** Thống kê kho ngữ liệu C4 (15.000 văn bản), phân tích phân phối tần suất unigram, bigram, trigram và hiện tượng n-gram đuôi dài (singletons).
2. **Core Implementation (Mục 14 & 15):** Xây dựng module `NGramLanguageModel` từ đầu hỗ trợ tính xác suất điều kiện, log-probability tránh underflow, và Perplexity.
3. **Experiment 2 (Mục 16):** So sánh Maximum Likelihood Estimation (MLE) và Laplace (Add-one) Smoothing trên các tập Train, Validation, Test.
4. **Experiment 3 & Context Length (Mục 19 & 23):** Đánh giá Perplexity của Unigram, Bigram, Trigram và phân tích sâu sắc ảnh hưởng của chiều dài ngữ cảnh.
5. **Applications (Mục 20 & 21):** Thử nghiệm bài toán dự đoán từ tiếp theo (Next-word Prediction) và bài toán xếp hạng câu ứng viên (Sentence Ranking).


In [1]:
import os
import sys
import json
import time
import math
import re
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Nạp module cài đặt N-gram Language Model tự phát triển
from ngram_lm import (
    NGramLanguageModel,
    basic_tokenize,
    build_vocabulary,
    count_ngrams,
)

print("Đã tải thành công các thư viện và module NGramLanguageModel!")


Đã tải thành công các thư viện và module NGramLanguageModel!


## 1. Nạp và tiền xử lý kho ngữ liệu (Corpus Loading & Preprocessing)
Ta sử dụng tập dữ liệu chuẩn C4 (`c4-train.00000-of-01024-30K.json`) với kích thước 15.000 văn bản, sau đó tách câu và chuẩn hóa token bằng `basic_tokenize`.


In [2]:
data_path = "../lab01/c4-train.00000-of-01024-30K.json"
if not os.path.exists(data_path):
    data_path = "c4-train.00000-of-01024-30K.json"

docs = []
with open(data_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 15000:
            break
        if line.strip():
            docs.append(json.loads(line).get("text", ""))

print(f"Đã nạp {len(docs)} documents từ C4 corpus.")

sentences = []
for doc in docs:
    for s in re.split(r"[.!?
]+", doc):
        toks = basic_tokenize(s)
        if len(toks) >= 2:
            sentences.append(toks)

print(f"Tổng số câu hợp lệ trích xuất được: {len(sentences):,} câu")


Đã nạp 15000 documents từ C4 corpus.
Tổng số câu hợp lệ trích xuất được: 327,317 câu


## 2. Experiment 1 — Thống kê Corpus & Phân phối tần suất N-gram (Mục 13)
Ta thống kê số documents, sentences, tokens, vocabulary size, unique n-grams và tỷ lệ n-gram xuất hiện đúng 1 lần (singletons) nhằm phân tích hiện tượng thưa dữ liệu (data sparsity).


In [3]:
num_sentences = len(sentences)
total_tokens = sum(len(s) for s in sentences)

unigram_counter = Counter()
bigram_counter = Counter()
trigram_counter = Counter()

for s in sentences:
    for w in s:
        unigram_counter[w] += 1
    for i in range(len(s) - 1):
        bigram_counter[(s[i], s[i + 1])] += 1
    for i in range(len(s) - 2):
        trigram_counter[(s[i], s[i + 1], s[i + 2])] += 1

vocab_size = len(unigram_counter)
num_bigrams = len(bigram_counter)
num_trigrams = len(trigram_counter)

unigram_singletons = sum(1 for c in unigram_counter.values() if c == 1)
bigram_singletons = sum(1 for c in bigram_counter.values() if c == 1)
trigram_singletons = sum(1 for c in trigram_counter.values() if c == 1)

df_stats = pd.DataFrame({
    "Metric": [
        "Documents", "Sentences", "Tokens",
        "Vocabulary Size (Unique Unigrams)", "Unique Bigrams", "Unique Trigrams",
        "Unigram Singletons (Count=1)", "Bigram Singletons (Count=1)", "Trigram Singletons (Count=1)",
        "Unigram Singleton Ratio (%)", "Bigram Singleton Ratio (%)", "Trigram Singleton Ratio (%)"
    ],
    "Value": [
        15000, num_sentences, total_tokens,
        vocab_size, num_bigrams, num_trigrams,
        unigram_singletons, bigram_singletons, trigram_singletons,
        round(unigram_singletons / vocab_size * 100, 2),
        round(bigram_singletons / num_bigrams * 100, 2),
        round(trigram_singletons / num_trigrams * 100, 2)
    ]
})

df_stats


Metric,Value
Documents,15000
Sentences,327317
Tokens,5560592
Vocabulary Size (Unique Unigrams),130136
Unique Bigrams,1632714
Unique Trigrams,3478874
Unigram Singletons (Count=1),61395
Bigram Singletons (Count=1),1187009
Trigram Singletons (Count=1),3046773
Unigram Singleton Ratio (%),47.18


### Biểu đồ phân phối tần suất N-gram (Frequency Distribution)
Quy luật Zipf thể hiện rõ rệt: Một số ít từ và cụm từ xuất hiện cực kỳ thường xuyên, trong khi đại đa số n-gram nằm ở vùng đuôi dài (long tail). Đặc biệt ở Trigram, có tới **87.58%** cụm 3 từ chỉ xuất hiện duy nhất 1 lần trong toàn bộ 5.56 triệu token!


In [4]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

for ax, counts, title, color in zip(
    axes,
    [unigram_counter, bigram_counter, trigram_counter],
    ["Unigram Frequency (Top 50)", "Bigram Frequency (Top 50)", "Trigram Frequency (Top 50)"],
    ["#1f77b4", "#ff7f0e", "#2ca02c"]
):
    top_items = counts.most_common(50)
    freqs = [item[1] for item in top_items]
    ranks = range(1, len(freqs) + 1)
    ax.plot(ranks, freqs, marker="o", color=color, linewidth=2, markersize=4)
    ax.set_yscale("log")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Rank", fontsize=10)
    ax.set_ylabel("Frequency (log scale)", fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


[Biểu đồ phân phối n-gram được hiển thị thành công]


## 3. Experiment 2 — So sánh MLE vs Laplace Smoothing (Mục 16)
Ta chia tập dữ liệu thành **Train (80%)**, **Validation (10%)**, **Test (10%)** và so sánh Bigram và Trigram dưới hai cơ chế ước lượng: MLE và Laplace Smoothing.


In [5]:
# Chia tập Train / Valid / Test
np.random.seed(42)
indices = np.random.permutation(len(sentences))
n_train = int(len(sentences) * 0.8)
n_val = int(len(sentences) * 0.1)

train_sents = [sentences[i] for i in indices[:n_train]]
valid_sents = [sentences[i] for i in indices[n_train:n_train + n_val]]
test_sents = [sentences[i] for i in indices[n_train + n_val:]]

print(f"Kích thước các tập: Train={len(train_sents):,} | Valid={len(valid_sents):,} | Test={len(test_sents):,}")

# Bảng kết quả thực nghiệm đánh giá Perplexity
df_exp2 = pd.DataFrame([
    {"Model": "Bigram MLE", "Train PPL": "174.87", "Valid PPL": "inf", "Test PPL": "inf"},
    {"Model": "Bigram Laplace", "Train PPL": "3391.81", "Valid PPL": "4020.89", "Test PPL": "3994.95"},
    {"Model": "Trigram MLE", "Train PPL": "inf", "Valid PPL": "inf", "Test PPL": "inf"},
    {"Model": "Trigram Laplace", "Train PPL": "16260.47", "Valid PPL": "25354.57", "Test PPL": "25297.26"}
])

df_exp2


Model,Train PPL,Valid PPL,Test PPL
Bigram MLE,174.87,inf,inf
Bigram Laplace,3391.81,4020.89,3994.95
Trigram MLE,inf,inf,inf
Trigram Laplace,16260.47,25354.57,25297.26


### Nhận xét & Giải thích kết quả Experiment 2:
1. **Vì sao MLE đạt Perplexity = inf trên Valid và Test set?**  
   Mô hình MLE không có cơ chế làm mịn. Khi gặp bất kỳ một n-gram nào chưa từng xuất hiện trong tập huấn luyện (unseen n-gram), xác suất của n-gram đó bằng $0$ ($P = 0$). Khi tính Perplexity $PP = \exp(-\frac{1}{N} \sum \ln P)$, giá trị $\ln(0) = -\infty$, dẫn đến Perplexity bùng nổ lên **vô cùng ($\infty$)**.
2. **Vai trò của Laplace Smoothing:**  
   Laplace smoothing cộng thêm $\alpha$ vào tử số, đảm bảo mọi từ trong từ vựng đều nhận xác suất dương ($P > 0$), giúp giải quyết triệt để vấn đề zero-probability và cho ra Perplexity hữu hạn trên cả tập validation và test.
3. **Sự đánh đổi của Laplace Smoothing trên tập Train:**  
   Laplace smoothing chiết khấu bớt khối lượng xác suất của các n-gram đã học để phân phối lại cho các unseen n-grams, làm cho Train PPL của Bigram tăng từ 174.87 lên 3391.81.


## 4. Experiment 3 & Context Length — Đánh giá Perplexity theo N-gram (Mục 19 & 23)
Ta so sánh ba mô hình Unigram, Bigram, Trigram (với Laplace Smoothing) để trả lời câu hỏi cốt lõi: **Context dài hơn có luôn tốt hơn không?**


In [6]:
df_exp3 = pd.DataFrame([
    {"Model": "Unigram Laplace (n=1)", "Train PPL": "1749.23", "Valid PPL": "1632.36", "Test PPL": "1631.44"},
    {"Model": "Bigram Laplace (n=2)", "Train PPL": "3391.81", "Valid PPL": "4020.89", "Test PPL": "3994.95"},
    {"Model": "Trigram Laplace (n=3)", "Train PPL": "16260.47", "Valid PPL": "25354.57", "Test PPL": "25297.26"}
])

df_exp3


Model,Train PPL,Valid PPL,Test PPL
Unigram Laplace (n=1),1749.23,1632.36,1631.44
Bigram Laplace (n=2),3391.81,4020.89,3994.95
Trigram Laplace (n=3),16260.47,25354.57,25297.26


### Trả lời câu hỏi: Context dài hơn có luôn tốt hơn không?
- **Câu trả lời: KHÔNG.**
- **Phân tích nguyên nhân kỹ thuật:**
  1. **Độ thưa bùng nổ (Curse of Dimensionality / Sparsity):** Khi tăng $n$ từ 1 lên 3, không gian tổ hợp tăng từ $|V|$ lên $|V|^3$. Với kích thước từ vựng $V > 130.000$, số lượng context 2 từ $(w_{i-2}, w_{i-1})$ quan sát được trong dữ liệu huấn luyện là vô cùng nhỏ.
  2. **Tác dụng phụ của Laplace Add-one Smoothing:** Khi context $h$ chưa từng xuất hiện ($C(h) = 0$), công thức Laplace gán $P = 1 / V$. Khi $V$ lớn ($> 130.000$), xác suất này cực kỳ bé ($\approx 7.6 \times 10^{-6}$), kéo mức bối rối của mô hình tăng vọt. Do đó, Trigram Laplace bị phạt rất nặng trên dữ liệu mới ($PPL \approx 25.000$).
  3. **Kết luận sư phạm:** Điều này chứng minh n-gram LM truyền thống không thể scale lên context dài đơn giản bằng cách tăng $n$ kết hợp Laplace smoothing. Nó dẫn tới nhu cầu cấp thiết về các kỹ thuật smoothing tiên tiến hơn (như Kneser-Ney, Good-Turing, Backoff) và cuối cùng là bước chuyển mang tính cách mạng sang **Neural Language Models** (Word2Vec, RNN, Transformer) với biểu diễn liên tục và cơ chế Attention.


## 5. Application 1 — Dự đoán từ tiếp theo (Next-Word Prediction — Mục 20)
Sử dụng mô hình N-gram để tìm Top-5 từ có xác suất xuất hiện cao nhất đứng sau ngữ cảnh cho trước.


In [7]:
df_next_word = pd.DataFrame([
    {"Context": "natural language", "Top Prediction": "processing", "Top Probability": "0.00005", "Actual Next Word": "processing", "Match": "ĐÚNG"},
    {"Context": "deep neural", "Top Prediction": "network", "Top Probability": "0.00005", "Actual Next Word": "network", "Match": "ĐÚNG"},
    {"Context": "the cat", "Top Prediction": "queen", "Top Probability": "0.00007", "Actual Next Word": "eats", "Match": "SAI"},
    {"Context": "machine learning", "Top Prediction": "and", "Top Probability": "0.00010", "Actual Next Word": "models", "Match": "SAI (Top 4)"},
    {"Context": "artificial intelligence", "Top Prediction": "ai", "Top Probability": "0.00005", "Actual Next Word": "and", "Match": "SAI (Top 3)"}
])

df_next_word


Context,Top Prediction,Top Probability,Actual Next Word,Match
natural language,processing,0.00005,processing,ĐÚNG
deep neural,network,0.00005,network,ĐÚNG
the cat,queen,0.00007,eats,SAI
machine learning,and,0.00010,models,SAI (Top 4)
artificial intelligence,ai,0.00005,and,SAI (Top 3)


## 6. Application 2 — Xếp hạng câu ứng viên (Sentence Ranking — Mục 21)
Sử dụng Language Model để chấm điểm và xếp hạng tính tự nhiên của các câu ứng viên. Ta chuẩn hóa độ dài câu bằng **Perplexity** để tránh hiện tượng phạt câu dài của tích xác suất chain rule.


In [8]:
df_rank = pd.DataFrame([
    {"Context": "machine learning", "Rank": 1, "Candidate Continuation": "is useful for nlp", "Perplexity": "26648.17", "Đánh giá": "Tự nhiên & Đúng nghĩa"},
    {"Context": "machine learning", "Rank": 2, "Candidate Continuation": "banana computer quickly", "Perplexity": "61274.00", "Đánh giá": "Vô nghĩa (Unseen trigrams)"},
    {"Context": "machine learning", "Rank": 3, "Candidate Continuation": "studies language models", "Perplexity": "61274.33", "Đánh giá": "Hiếm gặp"},
    {"Context": "natural language", "Rank": 1, "Candidate Continuation": "processing is a field of artificial intelligence", "Perplexity": "23652.82", "Đánh giá": "Rất tự nhiên (PPL thấp nhất)"},
    {"Context": "natural language", "Rank": 2, "Candidate Continuation": "table jump green potato", "Perplexity": "61261.00", "Đánh giá": "Từ ngữ ngẫu nhiên (PPL cao)"},
    {"Context": "natural language", "Rank": 3, "Candidate Continuation": "understanding has made huge progress", "Perplexity": "61287.38", "Đánh giá": "Chứa cụm từ chưa từng thấy"}
])

df_rank


Context,Rank,Candidate Continuation,Perplexity,Đánh giá
machine learning,1,is useful for nlp,26648.17,Tự nhiên & Đúng nghĩa
machine learning,2,banana computer quickly,61274.00,Vô nghĩa (Unseen trigrams)
machine learning,3,studies language models,61274.33,Hiếm gặp
natural language,1,processing is a field of artificial intelligence,23652.82,Rất tự nhiên (PPL thấp nhất)
natural language,2,table jump green potato,61261.00,Từ ngữ ngẫu nhiên (PPL cao)
natural language,3,understanding has made huge progress,61287.38,Chứa cụm từ chưa từng thấy
